### hybrid_deepseek.py
#### Hybrid inference with DeepSeek
This notebook evaluates:

1. Logistic Regression
2. Support Vector Machine
3. Naive Bayes
4. Random Forest
5. DL (BiLSTM)
6. Each baseline + DeepSeek refinement

This version is aligned with:
- `shared_preprocessing.ipynb`
- `train_ml.ipynb`
- `train_dl.ipynb`

So it uses the **same shared dataset**, **same shared split**, and **saved models** instead of training new models again inside the notebook.


In [27]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from openai import OpenAI
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

### Files

In [28]:
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE = "../data/splits/shared_split.json"

ML_VECTORIZER_FILE = "../models/ml_tfidf_vectorizer.joblib"
LR_MODEL_FILE = "../models/ml_models/logistic_regression.joblib"
SVM_MODEL_FILE = "../models/ml_models/svm.joblib"
NB_MODEL_FILE = "../models/ml_models/naive_bayes.joblib"
RF_MODEL_FILE = "../models/ml_models/random_forest.joblib"

DL_MODEL_FILE = "../models/bilstm_model.keras"
WORD_INDEX_FILE = "../models/word_index.json"
LABEL_MAPPING_FILE = "../models/label_mapping.json"
DL_METADATA_FILE = "../models/dl_metadata.json"

OUTPUT_FILE = "../data/processed/hybrid_deepseek_results.csv"
SUMMARY_FILE = "../data/processed/hybrid_deepseek_summary.csv"

os.makedirs("../data/processed", exist_ok=True)

### Load test set

In [29]:
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split_dict = json.load(f)

test_df = df[df["row_id"].isin(split_dict["test_ids"])].copy().reset_index(drop=True)

print("Test shape:", test_df.shape)
print(test_df["label"].value_counts())

valid_labels = sorted(df["label"].unique().tolist())
valid_labels

Test shape: (324, 3)
label
neutral    75
disgust    64
fear       60
angry      49
sad        41
happy      35
Name: count, dtype: int64


['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']

### Load ML vectorizer

In [30]:
ml_vectorizer = joblib.load(ML_VECTORIZER_FILE)
X_test_ml = ml_vectorizer.transform(test_df["text"]).toarray()
print("X_test_ml shape:", X_test_ml.shape)

X_test_ml shape: (324, 3000)


### Load all 4 ML models

In [31]:
lr_model = joblib.load(LR_MODEL_FILE)
svm_model = joblib.load(SVM_MODEL_FILE)
nb_model = joblib.load(NB_MODEL_FILE)
rf_model = joblib.load(RF_MODEL_FILE)

# Predict ML
lr_pred = lr_model.predict(X_test_ml)
lr_proba = lr_model.predict_proba(X_test_ml)
lr_conf = lr_proba.max(axis=1)

svm_pred = svm_model.predict(X_test_ml)
svm_proba = svm_model.predict_proba(X_test_ml)
svm_conf = svm_proba.max(axis=1)

nb_pred = nb_model.predict(X_test_ml)
nb_proba = nb_model.predict_proba(X_test_ml)
nb_conf = nb_proba.max(axis=1)

rf_pred = rf_model.predict(X_test_ml)
rf_proba = rf_model.predict_proba(X_test_ml)
rf_conf = rf_proba.max(axis=1)

print("ML predictions ready")

ML predictions ready


### Load DL model

In [32]:
dl_model = load_model(DL_MODEL_FILE)

with open(WORD_INDEX_FILE, "r", encoding="utf-8") as f:
    word_index = json.load(f)

with open(LABEL_MAPPING_FILE, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

with open(DL_METADATA_FILE, "r", encoding="utf-8") as f:
    dl_metadata = json.load(f)

tokenizer = Tokenizer(
    num_words=dl_metadata["max_vocab_size"],
    oov_token="<OOV>"
)
tokenizer.word_index = word_index

max_sequence_length = dl_metadata["max_sequence_length"]

X_test_seq = tokenizer.texts_to_sequences(test_df["text"])
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_sequence_length,
    padding="post",
    truncating="post"
)

index_to_label = {v: k for k, v in label_mapping.items()}

dl_proba = dl_model.predict(X_test_pad, verbose=0)
dl_pred_idx = np.argmax(dl_proba, axis=1)
dl_pred = [index_to_label[int(i)] for i in dl_pred_idx]
dl_conf = dl_proba.max(axis=1)

print("DL predictions ready")

DL predictions ready


### DeepSeek client

In [33]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

DEEPSEEK_MODEL = "deepseek-r1:8b"
print("AI ready:", DEEPSEEK_MODEL)

AI ready: deepseek-r1:8b


### Helper functions

In [34]:
def normalize_label(answer):
    answer = str(answer).strip().lower()

    for label in valid_labels:
        if answer == label.lower():
            return label

    for label in valid_labels:
        if label.lower() in answer:
            return label

    return ""

def ai_refine(text, baseline_pred, confidence, threshold=0.75):
    # If baseline is already confident enough, keep the baseline result
    if confidence >= threshold:
        return baseline_pred

    label_text = ", ".join(valid_labels)

    system_prompt = (
        "You are an emotion classification assistant. "
        "Choose exactly one label from the allowed labels only. "
        "Return only the label."
    )

    user_prompt = f"""
Allowed labels:
{label_text}

Patient utterance:
{text}

Baseline predicted label:
{baseline_pred}

Return exactly one label only.
""".strip()

    try:
        response = client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )

        answer = response.choices[0].message.content.strip()
        final_label = normalize_label(answer)

        if final_label == "":
            return baseline_pred

        return final_label

    except Exception as e:
        print("AI error:", e)
        return baseline_pred

def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    print(f"\n===== {model_name} =====")
    print(f"Accuracy     : {acc:.4f}")
    print(f"F1 Macro     : {f1_macro:.4f}")
    print(f"F1 Weighted  : {f1_weighted:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted
    }

### Build results table

In [35]:
results_df = test_df[["row_id", "text", "label"]].copy()

results_df["lr_pred"] = lr_pred
results_df["lr_conf"] = lr_conf

results_df["svm_pred"] = svm_pred
results_df["svm_conf"] = svm_conf

results_df["nb_pred"] = nb_pred
results_df["nb_conf"] = nb_conf

results_df["rf_pred"] = rf_pred
results_df["rf_conf"] = rf_conf

results_df["dl_pred"] = dl_pred
results_df["dl_conf"] = dl_conf

results_df.head()

,row_id,text,label,lr_pred,lr_conf,svm_pred,svm_conf,nb_pred,nb_conf,rf_pred,rf_conf,dl_pred,dl_conf
0,1,i keep thinking about how long ive been living...,disgust,disgust,0.630010,disgust,0.962754,disgust,0.546534,disgust,0.68,disgust,0.999164
1,14,i am extremely unhappy with this service,angry,angry,0.789079,angry,0.990640,angry,0.951720,angry,1.00,angry,0.998520
2,17,this has been exhausting for me and drilling i...,sad,sad,0.851704,sad,0.992580,sad,0.994491,sad,0.99,sad,0.998580
3,22,the idea of numbing injection is making me une...,fear,fear,0.709761,fear,0.934271,fear,0.953508,fear,0.76,fear,0.999844
4,24,i wanted to mention that i have been using the...,neutral,neutral,0.836092,neutral,0.964005,neutral,0.917400,neutral,0.98,neutral,0.983560


### Refine low-confidence predictions with DeepSeek

In [36]:
lr_ai_pred = []
svm_ai_pred = []
nb_ai_pred = []
rf_ai_pred = []
dl_ai_pred = []

for _, row in results_df.iterrows():
    lr_ai_pred.append(row["lr_pred"] if row["lr_conf"] >= 0.60 else ai_refine(row["text"], row["lr_pred"], row["lr_conf"]))
    svm_ai_pred.append(row["svm_pred"] if row["svm_conf"] >= 0.60 else ai_refine(row["text"], row["svm_pred"], row["svm_conf"]))
    nb_ai_pred.append(row["nb_pred"] if row["nb_conf"] >= 0.60 else ai_refine(row["text"], row["nb_pred"], row["nb_conf"]))
    rf_ai_pred.append(row["rf_pred"] if row["rf_conf"] >= 0.60 else ai_refine(row["text"], row["rf_pred"], row["rf_conf"]))
    dl_ai_pred.append(row["dl_pred"] if row["dl_conf"] >= 0.60 else ai_refine(row["text"], row["dl_pred"], row["dl_conf"]))

results_df["lr_ai_pred"] = lr_ai_pred
results_df["svm_ai_pred"] = svm_ai_pred
results_df["nb_ai_pred"] = nb_ai_pred
results_df["rf_ai_pred"] = rf_ai_pred
results_df["dl_ai_pred"] = dl_ai_pred

results_df.head()

,row_id,text,label,lr_pred,lr_conf,svm_pred,svm_conf,nb_pred,nb_conf,rf_pred,rf_conf,dl_pred,dl_conf,lr_ai_pred,svm_ai_pred,nb_ai_pred,rf_ai_pred,dl_ai_pred
0,1,i keep thinking about how long ive been living...,disgust,disgust,0.630010,disgust,0.962754,disgust,0.546534,disgust,0.68,disgust,0.999164,disgust,disgust,disgust,disgust,disgust
1,14,i am extremely unhappy with this service,angry,angry,0.789079,angry,0.990640,angry,0.951720,angry,1.00,angry,0.998520,angry,angry,angry,angry,angry
2,17,this has been exhausting for me and drilling i...,sad,sad,0.851704,sad,0.992580,sad,0.994491,sad,0.99,sad,0.998580,sad,sad,sad,sad,sad
3,22,the idea of numbing injection is making me une...,fear,fear,0.709761,fear,0.934271,fear,0.953508,fear,0.76,fear,0.999844,fear,fear,fear,fear,fear
4,24,i wanted to mention that i have been using the...,neutral,neutral,0.836092,neutral,0.964005,neutral,0.917400,neutral,0.98,neutral,0.983560,neutral,neutral,neutral,neutral,neutral


### Summary

In [37]:
summary_rows = []

summary_rows.append(evaluate_model(results_df["label"], results_df["lr_pred"], "Logistic Regression"))
summary_rows.append(evaluate_model(results_df["label"], results_df["svm_pred"], "Support Vector Machine"))
summary_rows.append(evaluate_model(results_df["label"], results_df["nb_pred"], "Naive Bayes"))
summary_rows.append(evaluate_model(results_df["label"], results_df["rf_pred"], "Random Forest"))
summary_rows.append(evaluate_model(results_df["label"], results_df["dl_pred"], "DL"))

summary_rows.append(evaluate_model(results_df["label"], results_df["lr_ai_pred"], "Logistic Regression + DeepSeek"))
summary_rows.append(evaluate_model(results_df["label"], results_df["svm_ai_pred"], "Support Vector Machine + DeepSeek"))
summary_rows.append(evaluate_model(results_df["label"], results_df["nb_ai_pred"], "Naive Bayes + DeepSeek"))
summary_rows.append(evaluate_model(results_df["label"], results_df["rf_ai_pred"], "Random Forest + DeepSeek"))
summary_rows.append(evaluate_model(results_df["label"], results_df["dl_ai_pred"], "DL + DeepSeek"))

summary_df = pd.DataFrame(summary_rows).sort_values(by="F1_Macro", ascending=False)

print("\n===== Summary =====")
print(summary_df)



===== Logistic Regression =====
Accuracy     : 0.9074
F1 Macro     : 0.9229
F1 Weighted  : 0.9075

Classification Report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        49
     disgust       0.87      0.92      0.89        64
        fear       0.84      0.82      0.83        60
       happy       1.00      0.97      0.99        35
     neutral       0.84      0.84      0.84        75
         sad       1.00      0.98      0.99        41

    accuracy                           0.91       324
   macro avg       0.93      0.92      0.92       324
weighted avg       0.91      0.91      0.91       324


Confusion Matrix:
[[49  0  0  0  0  0]
 [ 0 59  1  0  4  0]
 [ 0  4 49  0  7  0]
 [ 0  0  0 34  1  0]
 [ 0  5  7  0 63  0]
 [ 0  0  1  0  0 40]]

===== Support Vector Machine =====
Accuracy     : 0.9228
F1 Macro     : 0.9355
F1 Weighted  : 0.9229

Classification Report:
              precision    recall  f1-score   support

       

In [38]:
results_df = summary_df.copy()
results_df = results_df.sort_values(by="F1_Macro", ascending=False)
results_df = results_df.round(6)
results_df

,Model,Accuracy,F1_Macro,F1_Weighted
1,Support Vector Machine,0.922840,0.935528,0.922900
6,Support Vector Machine + DeepSeek,0.922840,0.931889,0.922521
0,Logistic Regression,0.907407,0.922935,0.907498
5,Logistic Regression + DeepSeek,0.913580,0.918849,0.912884
9,DL + DeepSeek,0.895062,0.911040,0.895075
8,Random Forest + DeepSeek,0.891975,0.901868,0.890576
7,Naive Bayes + DeepSeek,0.895062,0.899555,0.893460
3,Random Forest,0.876543,0.897599,0.876845
4,DL,0.870370,0.889321,0.869877
2,Naive Bayes,0.867284,0.880318,0.863803


### Save files

In [39]:
results_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
summary_df.to_csv(SUMMARY_FILE, index=False, encoding="utf-8-sig")

print("\nSaved files:")
print("-", OUTPUT_FILE)
print("-", SUMMARY_FILE)


Saved files:
- ../data/processed/hybrid_deepseek_results.csv
- ../data/processed/hybrid_deepseek_summary.csv
